# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

> **Quick Start:** Runtime → Run all (`Ctrl+F9`) · Record or upload audio — transcription starts automatically!

In [ ]:
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio


In [ ]:

import os, time, tempfile, json, re, urllib.request, gc
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (AVX2)"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = "float16" if device_type == "cuda" else "int8"
print(f"Hardware: {gpu_name} | {compute_dtype}")

_model_cache = {}
def get_model(name):
    if name not in _model_cache:
        os.makedirs("/content/models/whisper", exist_ok=True)
        _model_cache[name] = WhisperModel(name, device=device_type, compute_type=compute_dtype,
            num_workers=2, download_root="/content/models/whisper")
    return _model_cache[name]

print("Pre-warming large-v3-turbo…")
get_model("large-v3-turbo")
print("Pre-warming ECAPA-TDNN…")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa", run_opts={"device": device_type})
print("Ready.")

_roman = {'آپ':'aap','کیسے':'kaisay','ہیں':'hain','کیا':'kya','کر':'kar','رہے':'rahay',
    'ہو':'ho','میں':'main','ہوں':'hoon','یہ':'yeh','وہ':'woh','نہیں':'nahi',
    'ٹھیک':'theek','شکریہ':'shukriya','سلام':'salam','بہت':'bohot','اچھا':'acha'}
def to_roman(t):
    return " ".join(_roman.get(re.sub(r'[\u064B-\u065F\u0670]','',w),w) for w in t.split())

def load_16k(path):
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        return wav.squeeze().numpy().astype(np.float32)
    except Exception:
        d, sr = sf.read(path)
        if d.ndim > 1: d = d.mean(1)
        d = d.astype(np.float32)
        if sr != 16000:
            n = int(len(d)*16000/sr)
            d = np.interp(np.linspace(0,len(d),n,endpoint=False),np.arange(len(d)),d).astype(np.float32)
        return d

COLORS = ['#6366F1','#10B981','#F59E0B','#EC4899','#06B6D4','#8B5CF6','#F97316']
MODEL_MAP = {
    'Large-v3-Turbo (809M)': 'large-v3-turbo',
    'Whisper Large-v3 (1.5B)': 'large-v3',
    'Whisper Base (74M)': 'base',
    'Whisper Tiny (39M)': 'tiny',
    'Whisper Small (244M)': 'small',
    'Whisper Medium (769M)': 'medium',
}
LANG_MAP = {
    'Bilingual (Urdu + English)': (None, False),
    'Pure Urdu Script (اردو)': ('ur', False),
    'English Only': ('en', False),
    'Roman Urdu (Latin)': ('ur', True),
}

def transcribe(audio_path, model_choice, lang_choice, thresh_pct, vad_ms):
    EMPTY_TRANSCRIPT = """<div class='vd-empty'>
      <svg width='48' height='48' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.2' opacity='.25'>
        <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
      </svg>
      <p>Lecture transcript will appear here</p>
      <span>Record audio or upload a file to begin</span>
    </div>"""
    EMPTY_SIDEBAR = """<div class='vd-empty-sm'>
      <svg width='32' height='32' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.2' opacity='.25'>
        <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
        <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
      </svg>
      <p>No speakers detected</p>
    </div>"""

    if not audio_path or not os.path.exists(audio_path):
        return EMPTY_TRANSCRIPT, EMPTY_SIDEBAR, "", None, None, None, None

    t0 = time.time()
    data = load_16k(audio_path)
    dur = len(data)/16000.0
    mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
    model = get_model(mkey)
    target_lang, is_roman = LANG_MAP.get(lang_choice, (None, False))
    thresh = float(thresh_pct)/100.0

    segs, info = model.transcribe(data, beam_size=1, best_of=1, temperature=0.0,
        language=target_lang, without_timestamps=False, vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=int(vad_ms)))

    profiles = {}; nxt = 1
    html_parts = []; plain = []; srt_parts = []; json_arr = []; si = 1

    for seg in segs:
        txt = seg.text.strip()
        if not txt: continue
        if is_roman: txt = to_roman(txt)
        s0, s1 = seg.start, seg.end
        chunk = data[int(s0*16000):int(s1*16000)]
        spk = 1
        if len(chunk) >= 8000:
            try:
                with torch.inference_mode():
                    w = torch.from_numpy(chunk).float().unsqueeze(0).to(device_type)
                    e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                    en = e/(np.linalg.norm(e) or 1.)
                bid, bsim = None, -1.
                for sid, embs in profiles.items():
                    ms_ = max(float(np.dot(en,x)) for x in embs) if embs else 0
                    if ms_ > bsim: bsim, bid = ms_, sid
                if bid and bsim >= thresh:
                    spk = bid
                    if len(profiles[spk]) < 50: profiles[spk].append(en)
                else:
                    spk = nxt; profiles[spk] = [en]; nxt += 1
            except: pass

        c = COLORS[(spk-1)%len(COLORS)]
        ts = f"{int(s0//60):02d}:{int(s0%60):02d}"
        urdu = any('\u0600' <= ch <= '\u06FF' for ch in txt)
        txt_class = "vd-txt vd-rtl" if urdu else "vd-txt"

        html_parts.append(f"""<div class="vd-seg">
  <div class="vd-seg-bar" style="background:{c}"></div>
  <div class="vd-seg-body">
    <div class="vd-seg-meta">
      <span class="vd-spk-lbl" style="color:{c}">Speaker {spk}</span>
      <span class="vd-time">{ts}</span>
    </div>
    <div class="{txt_class}">{txt}</div>
  </div>
</div>""")
        plain.append(f"[{ts}] Speaker {spk}: {txt}")
        def srt_ts(s):
            return f"{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int((s-int(s))*1000):03d}"
        srt_parts.append(f"{si}\n{srt_ts(s0)} --> {srt_ts(s1)}\n[Speaker {spk}]: {txt}\n")
        json_arr.append({"speaker":f"Speaker {spk}","start":round(s0,2),"end":round(s1,2),"text":txt})
        si += 1

    elapsed = time.time() - t0

    # Speaker sidebar
    sb = []
    for sid, embs in profiles.items():
        cc = COLORS[(sid-1)%len(COLORS)]
        sb.append(f"""<div class="vd-spk-card">
  <div class="vd-spk-av" style="background:{cc};box-shadow:0 0 12px {cc}44">S{sid}</div>
  <div>
    <div class="vd-spk-name">Speaker {sid}</div>
    <div class="vd-spk-meta">{len(embs)} voiceprint{'s' if len(embs)!=1 else ''}</div>
  </div>
</div>""")
    if not sb:
        sb = ["<div class='vd-empty-sm'><p>No speakers detected</p></div>"]

    stats = f"""<div class="vd-stats">
  <span>{mkey} &nbsp;·&nbsp; {gpu_name}</span>
  <span>{dur:.1f}s audio &nbsp;→&nbsp; {elapsed:.1f}s &nbsp;({dur/max(.01,elapsed):.1f}× real-time)</span>
</div>"""

    full_transcript = "\n".join(html_parts) + stats

    # Export files
    files = {}
    for ext, content in [
        ('.md',   "# VoiceDiary Lecture Notes\n\n" + "\n\n".join(plain)),
        ('.txt',  "\n".join(plain)),
        ('.srt',  "\n".join(srt_parts)),
        ('.json', json.dumps(json_arr, indent=2, ensure_ascii=False))
    ]:
        tmp = tempfile.NamedTemporaryFile(mode='w', suffix=ext, delete=False,
                                          encoding='utf-8', prefix='VoiceDiary_')
        tmp.write(content); tmp.close()
        files[ext] = tmp.name

    export_html = ""
    if plain:
        export_html = f"""<div class="vd-export-row">
  <a class="vd-dl" href="/file={files['.md']}" download>
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/><line x1="12" y1="18" x2="12" y2="12"/><line x1="9" y1="15" x2="15" y2="15"/></svg>
    Markdown
  </a>
  <a class="vd-dl" href="/file={files['.txt']}" download>
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
    Plain Text
  </a>
  <a class="vd-dl" href="/file={files['.srt']}" download>
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="2" y="2" width="20" height="20" rx="2"/><path d="M8 10h8M8 14h5"/></svg>
    Subtitles .srt
  </a>
  <a class="vd-dl" href="/file={files['.json']}" download>
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="16 18 22 12 16 6"/><polyline points="8 6 2 12 8 18"/></svg>
    JSON Data
  </a>
</div>"""

    return full_transcript, "\n".join(sb), export_html, "\n".join(plain), files.get('.md'), files.get('.txt'), files.get('.srt')

def gemini_summary(text, key):
    if not text or not text.strip(): return "*No transcript to summarize yet.*"
    api_key = (key or "").strip() or os.environ.get("GEMINI_API_KEY","")
    if not api_key: return "*Add your Gemini API key in the sidebar to generate AI study notes.*"
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    payload = {"contents":[{"parts":[{"text":
        f"You are VoiceDiary AI. Analyze this diarized lecture transcript and produce:\n"
        f"1. **Executive Overview** (2-3 sentences)\n"
        f"2. **Core Concepts Covered**\n"
        f"3. **Key Points for Exam**\n"
        f"4. **Notable Q&A Highlights**\n\nTranscript:\n{text}"}]}]}
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.loads(r.read())["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e:
        return f"*Error: {e}*"

# ─── CSS ────────────────────────────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@400;500&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

/* ── FORCE FULL VIEWPORT ── */
html, body { margin:0 !important; padding:0 !important; height:100% !important; }

.gradio-container {
  max-width: 100% !important;
  width: 100% !important;
  min-height: 100vh !important;
  margin: 0 !important;
  padding: 0 !important;
  background: #080C14 !important;
  font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
}

/* Background radial glow */
.gradio-container::before {
  content: '';
  position: fixed; inset: 0; z-index: 0; pointer-events: none;
  background:
    radial-gradient(ellipse 70% 45% at 50% -10%, rgba(99,102,241,.18) 0%, transparent 65%),
    radial-gradient(ellipse 50% 35% at 85% 85%, rgba(139,92,246,.10) 0%, transparent 55%);
}

/* ── WIPE GRADIO CHROME ── */
.gradio-container > .main > .wrap { padding: 0 !important; }
.gradio-container .block,
.gradio-container .form,
.gradio-container .gap { box-shadow:none !important; border:none !important; background:transparent !important; }
.gr-group, .gr-box, .gr-panel { background:transparent !important; border:none !important; box-shadow:none !important; }
footer, .footer, .gr-footer { display:none !important; }

/* Inputs / dropdowns / text areas */
.gradio-container select,
.gradio-container input[type=text],
.gradio-container input[type=password],
.gradio-container textarea {
  background: rgba(255,255,255,.05) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  border-radius: 8px !important;
  color: #F8FAFC !important;
  font-family: inherit !important;
  font-size: 13px !important;
  padding: 9px 12px !important;
}
.gradio-container select:focus,
.gradio-container input:focus { border-color: rgba(99,102,241,.5) !important; outline:none !important; }

/* Labels */
.gradio-container label > span,
.gradio-container .label-wrap span { color: #64748B !important; font-size: 11px !important; font-weight: 700 !important; letter-spacing:.05em !important; text-transform:uppercase !important; }

/* Sliders */
input[type=range] { accent-color: #6366F1 !important; }
.gradio-container .wrap.svelte-1kftt5q { background:transparent !important; }

/* Tabs */
.tab-nav { background: transparent !important; border-bottom: 1px solid rgba(255,255,255,.08) !important; }
.tab-nav button {
  color: #64748B !important; font-weight:600 !important; font-size:13px !important;
  padding: 10px 20px !important; background:transparent !important;
  border:none !important; border-bottom: 2px solid transparent !important;
  border-radius:0 !important; transition: all .15s !important;
}
.tab-nav button.selected { color:#F8FAFC !important; border-bottom-color: #6366F1 !important; }
.tabitem { background:transparent !important; border:none !important; padding: 16px 0 0 0 !important; }

/* Audio component */
.gradio-container .audio { background: rgba(255,255,255,.03) !important; border: 1px solid rgba(255,255,255,.08) !important; border-radius:12px !important; }

/* Hide file outputs */
.gradio-container .file-preview { display:none !important; }

/* Buttons */
.gradio-container button.primary {
  background: linear-gradient(135deg,#6366F1,#8B5CF6) !important;
  color:#fff !important; border:none !important;
  font-weight:700 !important; font-size:14px !important;
  border-radius:10px !important; padding:13px 24px !important;
  box-shadow: 0 0 24px rgba(99,102,241,.30) !important;
  transition: all .2s !important; width:100% !important;
}
.gradio-container button.primary:hover { transform:translateY(-2px) !important; box-shadow:0 0 32px rgba(99,102,241,.50) !important; }
.gradio-container button.secondary {
  background: rgba(255,255,255,.05) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  color:#94A3B8 !important; border-radius:8px !important;
  font-weight:600 !important; font-size:13px !important;
  transition: all .15s !important;
}
.gradio-container button.secondary:hover { background:rgba(255,255,255,.09) !important; color:#F8FAFC !important; }

/* ── LAYOUT SHELL ── */
#vd-app {
  display:grid;
  grid-template-columns: 280px 1fr;
  grid-template-rows: 64px 1fr;
  min-height: 100vh;
  gap: 0;
}

/* HEADER */
.vd-hdr {
  grid-column: 1/-1;
  display:flex; align-items:center; justify-content:space-between;
  padding: 0 28px;
  background: rgba(15,23,42,.85);
  backdrop-filter: blur(24px);
  border-bottom: 1px solid rgba(255,255,255,.07);
  position:sticky; top:0; z-index:100;
}
.vd-hdr-left { display:flex; align-items:center; gap:14px; }
.vd-logo-box {
  width:38px; height:38px; border-radius:10px; flex-shrink:0;
  background: linear-gradient(135deg,#6366F1,#8B5CF6);
  display:flex; align-items:center; justify-content:center;
  box-shadow: 0 0 16px rgba(99,102,241,.40);
}
.vd-hdr-title { font-size:18px; font-weight:800; color:#F8FAFC; letter-spacing:-.025em; line-height:1; }
.vd-hdr-sub { font-size:11px; color:#475569; font-weight:500; margin-top:2px; }
.vd-hw-pill {
  display:inline-flex; align-items:center; gap:8px;
  padding:6px 16px; border-radius:9999px;
  background: rgba(16,185,129,.10); border:1px solid rgba(16,185,129,.22);
  font-size:11px; font-weight:700; color:#10B981;
  font-family:'Fira Code',monospace; letter-spacing:.02em;
}
.vd-hw-dot { width:7px; height:7px; border-radius:50%; background:#10B981; box-shadow:0 0 8px rgba(16,185,129,.7); flex-shrink:0; }

/* SIDEBAR */
.vd-sidebar {
  background: rgba(15,23,42,.55);
  backdrop-filter: blur(20px);
  border-right: 1px solid rgba(255,255,255,.07);
  overflow-y: auto;
  padding: 24px 18px;
  display:flex; flex-direction:column; gap:28px;
}

/* Section label */
.vd-sec-lbl {
  font-size:10px; font-weight:800; color:#475569;
  letter-spacing:.10em; text-transform:uppercase;
  display:flex; align-items:center; justify-content:space-between;
  margin-bottom:14px;
}
.vd-badge {
  font-size:10px; font-weight:700; padding:2px 8px; border-radius:9999px;
  background: rgba(99,102,241,.15); color:#818CF8;
}
.vd-badge-live {
  width:6px; height:6px; border-radius:50%; background:#10B981;
  box-shadow:0 0 8px rgba(16,185,129,.8); display:inline-block; margin-right:4px;
  animation: pulse 2s infinite;
}
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.4} }

/* Speaker cards */
.vd-spk-card {
  display:flex; align-items:center; gap:12px;
  padding:11px 14px; border-radius:12px;
  background:rgba(255,255,255,.03); border:1px solid rgba(255,255,255,.07);
  margin-bottom:8px; transition:.15s ease;
}
.vd-spk-card:hover { background:rgba(255,255,255,.06); border-color:rgba(255,255,255,.12); transform:translateX(2px); }
.vd-spk-av {
  width:36px; height:36px; border-radius:50%; flex-shrink:0;
  display:flex; align-items:center; justify-content:center;
  font-weight:800; font-size:13px; color:#fff;
}
.vd-spk-name { font-size:13px; font-weight:600; color:#E2E8F0; }
.vd-spk-meta { font-size:11px; color:#475569; margin-top:1px; }

/* Empty states */
.vd-empty {
  display:flex; flex-direction:column; align-items:center; justify-content:center;
  padding:80px 32px; text-align:center; color:#334155; gap:12px;
}
.vd-empty p { font-size:16px; font-weight:600; color:#475569; margin:0; }
.vd-empty span { font-size:12px; color:#334155; }
.vd-empty-sm {
  display:flex; flex-direction:column; align-items:center;
  padding:28px 16px; text-align:center; color:#334155; gap:8px;
}
.vd-empty-sm p { font-size:13px; font-weight:600; color:#475569; margin:0; }

/* MAIN CONTENT */
.vd-main {
  display:flex; flex-direction:column; gap:0;
  overflow:hidden; background:transparent;
  padding: 24px 28px 28px 28px;
  overflow-y:auto;
}

/* Transcript viewport */
.vd-transcript {
  flex:1; overflow-y:auto;
  padding-right:4px;
  min-height:380px;
  max-height:calc(100vh - 380px);
}

/* Segment cards */
.vd-seg {
  display:flex; gap:0;
  margin-bottom:12px; border-radius:12px;
  background:rgba(15,23,42,.70); border:1px solid rgba(255,255,255,.07);
  overflow:hidden; transition:.15s ease;
}
.vd-seg:hover { background:rgba(30,41,59,.80); border-color:rgba(255,255,255,.12); }
.vd-seg-bar { width:4px; flex-shrink:0; }
.vd-seg-body { padding:13px 16px; flex:1; }
.vd-seg-meta { display:flex; align-items:center; gap:10px; margin-bottom:5px; }
.vd-spk-lbl { font-size:12px; font-weight:700; }
.vd-time { font-size:11px; color:#475569; font-family:'Fira Code',monospace; }
.vd-txt { font-size:15px; line-height:1.68; color:#CBD5E1; word-break:break-word; }
.vd-rtl { direction:rtl; text-align:right; font-family:'Noto Nastaliq Urdu',serif; font-size:17px; line-height:2.15; color:#E2E8F0; }

/* Stats bar */
.vd-stats {
  margin-top:14px; padding:10px 0; border-top:1px solid rgba(255,255,255,.07);
  display:flex; justify-content:space-between; align-items:center;
  font-size:11px; color:#475569; font-family:'Fira Code',monospace;
}

/* Export row */
.vd-export-row {
  display:flex; gap:10px; flex-wrap:wrap; margin-top:10px;
}
.vd-dl {
  display:inline-flex; align-items:center; gap:8px;
  padding:9px 16px; border-radius:8px;
  background:rgba(255,255,255,.04); border:1px solid rgba(255,255,255,.09);
  color:#94A3B8; font-size:13px; font-weight:600;
  text-decoration:none; transition:.15s ease; cursor:pointer;
}
.vd-dl:hover { background:rgba(99,102,241,.12); border-color:rgba(99,102,241,.35); color:#A5B4FC; }
.vd-dl svg { flex-shrink:0; opacity:.8; }

/* Divider between sections */
.vd-divider { border:none; border-top:1px solid rgba(255,255,255,.07); margin:20px 0; }
"""

with gr.Blocks(title="VoiceDiary — AI Lecture Engine", css=CSS,
               theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:

    transcript_state = gr.State("")

    # ── HEADER (HTML) ──
    gr.HTML(f"""
    <div class="vd-hdr">
      <div class="vd-hdr-left">
        <div class="vd-logo-box">
          <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.2">
            <path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/>
            <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
            <line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/>
          </svg>
        </div>
        <div>
          <div class="vd-hdr-title">VoiceDiary</div>
          <div class="vd-hdr-sub">AI Bilingual Lecture &amp; Diarization Engine</div>
        </div>
      </div>
      <div class="vd-hw-pill">
        <span class="vd-hw-dot"></span>
        {gpu_name} &nbsp;·&nbsp; Tensor Cores {compute_dtype.upper()}
      </div>
    </div>""")

    # ── MAIN TWO-COLUMN LAYOUT ──
    with gr.Row(equal_height=False):

        # ── LEFT SIDEBAR ──
        with gr.Column(scale=3, min_width=260):
            # Speakers
            gr.HTML("<div class='vd-sec-lbl'><span>Speakers &amp; Profiles</span><span><span class='vd-badge-live'></span>LIVE</span></div>")
            sidebar_out = gr.HTML(value="""<div class='vd-empty-sm'>
              <svg width='30' height='30' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.2' opacity='.3'>
                <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
                <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
              </svg>
              <p>No speakers detected</p>
            </div>""")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Engine &amp; Model Hub</span></div>")

            model_dd = gr.Dropdown(choices=list(MODEL_MAP.keys()),
                value='Large-v3-Turbo (809M)', label='Active Whisper Model')
            lang_dd = gr.Dropdown(choices=list(LANG_MAP.keys()),
                value='Bilingual (Urdu + English)', label='Language Output Mode')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Diarization Hyperparameters</span></div>")
            thresh_sl = gr.Slider(20, 70, 32, step=1, label='Speaker Similarity Threshold (%)')
            vad_sl = gr.Slider(150, 600, 280, step=10, label='VAD Silence Gap (ms)')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Gemini AI Key (BYOK)</span></div>")
            gemini_key = gr.Textbox(placeholder='sk-… or AIza…', type='password', label='', container=False)
            ai_btn = gr.Button("Generate AI Study Summary", variant="secondary")

        # ── RIGHT MAIN ──
        with gr.Column(scale=9, min_width=500):
            # Recording / upload tabs
            with gr.Tabs():
                with gr.TabItem("Live Classroom Lecture"):
                    audio_mic = gr.Audio(sources=["microphone"], type="filepath",
                                         label="Click microphone to record",
                                         show_label=False)
                with gr.TabItem("Upload Audio File"):
                    audio_file = gr.Audio(sources=["upload"], type="filepath",
                                          label="Drop or browse audio file (.wav .mp3 .m4a .flac)",
                                          show_label=False)

            transcribe_btn = gr.Button("Transcribe & Diarize Lecture  (GPU Accelerated)",
                                        variant="primary")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Classroom Lecture Transcript</span></div>")
            transcript_out = gr.HTML(value="""<div class='vd-empty'>
              <svg width='48' height='48' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.2' opacity='.2'>
                <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
              </svg>
              <p>Lecture transcript will appear here</p>
              <span>Record audio or upload a file to begin</span>
            </div>""")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Export Notes</span></div>")
            export_html_out = gr.HTML(value="<div style='font-size:12px;color:#334155;padding:4px 0'>Export options appear after transcription.</div>")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Study Summary &amp; Flashcards</span></div>")
            ai_out = gr.Markdown(value="*Generate AI summary using the button in the sidebar.*")

            # Hidden file refs for Gradio serving
            with gr.Row(visible=False):
                f_md  = gr.File()
                f_txt = gr.File()
                f_srt = gr.File()

    # ── EVENT WIRING ──
    all_inputs  = [audio_mic, audio_file, model_dd, lang_dd, thresh_sl, vad_sl]
    all_outputs = [transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt]

    def run_mic(mic, _f, mod, lang, th, vad): return transcribe(mic, mod, lang, th, vad)
    def run_btn(mic, fpath, mod, lang, th, vad): return transcribe(mic if mic else fpath, mod, lang, th, vad)

    audio_mic.stop_recording(fn=run_mic,   inputs=all_inputs, outputs=all_outputs)
    transcribe_btn.click(   fn=run_btn,    inputs=all_inputs, outputs=all_outputs)
    ai_btn.click(fn=gemini_summary, inputs=[transcript_state, gemini_key], outputs=[ai_out])

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
